In [ ]:
#@title << Run this cell first — environment setup {display-mode: "form"}
import sys, os

if "google.colab" in sys.modules:
    !git clone --quiet --single-branch --branch main https://github.com/YanickSchraner/CAS-DeepRL.git
    !cp -r "CAS-DeepRL/Tag 2/envs" .
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

# Unit: Advantage Actor-Critic (A2C) — Smart Building HVAC Control 🏢🌡️

## The Real-World Problem

Buildings account for **~40% of global energy consumption**, and heating/cooling is the largest single component. Smart thermostats and HVAC controllers are one of the most commercially viable RL applications today.

In **2016, Google DeepMind applied RL to the cooling systems of Google's data centers** and achieved a **40% reduction in cooling energy consumption**. The agent learned to pre-cool when electricity is cheap and reduce cooling during peak-price hours — something rule-based systems couldn't do efficiently.

**Your task:** Train an A2C agent to control a room heater. The agent must:
- Keep the room temperature close to **21°C** (comfort target)
- Minimize **electricity cost** (prices vary throughout the day — cheap at night, expensive in the evening)
- These two objectives are in tension: heating is cheapest when you least need it

### 📚 RL Library: [Stable-Baselines3](https://stable-baselines3.readthedocs.io/)

## Objectives 🏆

By the end of this notebook you will:
- Understand the A2C algorithm and when to use it
- Train an RL agent on a **custom gymnasium environment** with domain-relevant dynamics
- **Investigate the effect of hyperparameters** on training
- **Redesign the reward function** to change agent behaviour
- Diagnose a **poorly-trained agent** and explain why it failed

## Install and import

In [ ]:
!pip install stable-baselines3[extra] gymnasium matplotlib --quiet

In [ ]:
import sys, os
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from envs.hvac_env import HvacEnv

## Part 1 — Explore the Environment

Before training, always **understand your environment**. Let's lay out the **five MDP components** in building-control terms — the same discipline you used for the Day 1 inventory problem — and then watch a random agent.

---

### The Environment

You control the heater of a single room. Every **15 minutes** you pick a heater power level. A simple **RC thermal model** governs how the room warms (from the heater) and cools (heat leaking to the colder outdoors). The outdoor temperature follows a daily cycle, and the **electricity price** changes through the day — cheap overnight, peaking in the early evening. Small random fluctuations keep the dynamics noisy.

---

### Observation Space — what the agent sees each step

| Signal | Range | Meaning |
|---|---|---|
| `room_temp` | [10, 35] °C | Current room temperature — the thing you control |
| `outdoor_temp` | [-10, 35] °C | Outside temperature — drives how fast heat leaks away |
| `hour_sin` | [-1, 1] | Sine-encoded hour of day |
| `hour_cos` | [-1, 1] | Cosine-encoded hour of day (paired with `hour_sin` to pinpoint the time without a midnight discontinuity) |
| `price_norm` | [0, 1] | Current electricity price, normalised (0 = cheapest, 1 = peak) |

These five signals are **continuous**. On Day 1 you binned the inventory state into a Q-table; doing that here across 5 continuous dimensions would explode into billions of states. That is exactly why A2C uses a **neural network** to read the raw vector — the heart of *deep* RL.

---

### Actions — what the agent decides each step

The agent picks one of **4 discrete heater settings**:

| Action | Setting | Power |
|---|---|---|
| 0 | OFF | 0.0 kW |
| 1 | LOW | 0.5 kW |
| 2 | MEDIUM | 1.5 kW |
| 3 | HIGH | 3.0 kW |

More power warms the room faster but costs more — and costs *much* more during the evening price peak.

---

### Reward Signal — what we optimise

Each 15-minute step earns:

```
reward = −(discomfort_penalty + cost_weight × electricity_cost)
```

| Component | Formula | Intuition |
|---|---|---|
| Discomfort penalty | \|room_temp − 21 °C\| × 2.0 | The further from 21 °C, the unhappier the occupants |
| Electricity cost | heater_kW × price_€/kWh × 0.25 h | Energy used this step × its price |
| `cost_weight` | 10.0 (default) | Scales the cost into the same range as discomfort, so saving money is genuinely worth it |

The reward is always ≤ 0 — you are minimising a *cost*. The two terms are in **direct tension**: heat is cheapest overnight, exactly when you need it least. A good agent keeps the room near 21 °C **and** shifts heating into cheap hours.

---

### Episode — how long does one run last?

One episode = **one day = 96 steps** (24 h × 4 steps/h). The room starts each episode at a random temperature (16–26 °C) with a random outdoor baseline. There is no early termination; the agent maximises total reward over the full day.

Run the next cell to inspect the spaces, then watch a **random** agent — the baseline you must beat.

In [ ]:
env = HvacEnv()
check_env(env)  # validates the environment follows the gymnasium API

print("=== HVAC Environment ===")
print(f"Observation space : {env.observation_space}")
print(f"  [room_temp, outdoor_temp, hour_sin, hour_cos, price_norm]")
print(f"Action space      : {env.action_space}  (0=OFF, 1=LOW 0.5kW, 2=MED 1.5kW, 3=HIGH 3.0kW)")
print(f"Episode length    : {env.episode_length} steps (= 1 day in 15-min intervals)")
print()

# Sample one observation
obs, _ = env.reset(seed=42)
print("Sample observation:", obs)
print("  room_temp    :", obs[0], "°C")
print("  outdoor_temp :", obs[1], "°C")
print("  hour_sin/cos :", obs[2], obs[3])
print("  price_norm   :", obs[4], "(0=cheapest, 1=most expensive)")

In [ ]:
# Run a random agent for one episode and track temperature + reward
env = HvacEnv(seed=42)
obs, _ = env.reset()

temps, actions_taken, rewards_list, hours = [], [], [], []
done = False
hour = 0.0

while not done:
    action = env.action_space.sample()  # random action
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    temps.append(info["room_temp"])
    actions_taken.append(info["heater_kw"])
    rewards_list.append(reward)
    hour += 0.25
    hours.append(hour)

print(f"Random agent total reward: {sum(rewards_list):.1f}")
print(f"Average room temp: {np.mean(temps):.1f}°C  (target: 21°C)")
print(f"Time outside comfort zone (|T-21|>1): {np.mean(np.abs(np.array(temps)-21)>1)*100:.0f}% of steps")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(hours, temps, label="Room temp")
ax1.axhline(21, color="green", linestyle="--", label="Target 21°C")
ax1.fill_between(hours, 20, 22, alpha=0.1, color="green", label="Comfort zone")
ax1.set_ylabel("Temperature (°C)")
ax1.legend()
ax1.set_title("Random Agent — Room Temperature")

ax2.bar(hours, actions_taken, width=0.2, label="Heater power (kW)")
ax2.set_ylabel("Heater power (kW)")
ax2.set_xlabel("Hour of day")
ax2.legend()

plt.tight_layout()
plt.show()

**Observation:** The random agent switches the heater on and off erratically. Temperature fluctuates widely. This is your baseline — the agent you need to beat.

❓ **Question before training:** What strategy would *you* use if you were the controller? When would you heat more? When less?

In [ ]:
#@title 💡 Sample answer — what strategy would you use? (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
You want to keep the room near **21 °C** while buying electricity when it's cheap:

- **Pre-heat during the cheap overnight / early-morning trough**, nudging the room a little above 21 °C so the building's **thermal mass** stores heat. Then you can **ease off during the expensive evening peak** and let the stored heat coast you through it. This is exactly **load shifting** — the idea behind demand-response programs and DeepMind's data-centre result.
- **Heat less when it's mild outside** (less heat leaks away) and **more when it's cold** (faster heat loss).
- Avoid the naive thermostat trap of heating *reactively* whenever the temperature dips — that buys expensive peak electricity precisely when you should be coasting.

Keep this prediction in mind: after training, check whether the agent actually discovered pre-heating, or whether it just behaves like a price-blind thermostat.
"""))

## Part 2 — A2C: The Algorithm

**Advantage Actor-Critic (A2C)** combines two networks:
- **Actor**: outputs a probability distribution over actions (the *policy*)
- **Critic**: estimates the value function V(s) — how good is this state?

The **Advantage** A(s,a) = Q(s,a) - V(s) answers: *"Is this action better or worse than average in this state?"*
- A > 0 → this action led to better-than-expected return → increase its probability
- A < 0 → worse than expected → decrease its probability

**Why A2C over REINFORCE?**  
REINFORCE uses the full discounted return G_t as the signal, which is very noisy (high variance). The critic's value estimate V(s) acts as a **baseline** that reduces variance while keeping the gradient unbiased.

**Why A2C over Q-Learning/DQN?**  
A2C works natively with continuous and high-dimensional action spaces. DQN requires a discrete action space and can't scale as easily to complex policies.

## Part 3 — Train with Stable-Baselines3

### ⚠️ First, the most important RL-engineering habit: **normalise**

Look at the raw observation: `room_temp` lives in **[10, 35]** and `outdoor_temp` in **[−10, 35]**, while `hour_sin`, `hour_cos`, `price_norm` are all in **[−1, 1]**. Feed that straight into a neural network and two things go wrong:

1. **The big features dominate.** The temperature inputs (magnitude ~30) swamp the price/time signals (magnitude ~1) and **saturate the policy's `tanh` layers**, so the agent can barely *see* electricity price — exactly the signal it needs for load-shifting.
2. **The reward scale is huge.** Episode returns are around **−800 to −2000**, so the critic has to fit targets in the thousands. Watch `value_loss` (in the thousands) and `explained_variance` (stuck near 0): the critic never learns, the advantages are garbage, and the policy collapses to a single action — usually **heater permanently OFF**, which is *worse than random*.

The standard fix is **`VecNormalize`**, which keeps a running mean/std and normalises observations (and, optionally, rewards) on the fly. Two gotchas it forces you to handle, both good lessons:
- The **evaluation env must share the training statistics** (and not update them itself).
- At inference you must **normalise the observation before calling `model.predict`** — the helpers below do this for you.

Run the helper cell, then train.

In [ ]:
# --- Normalisation helpers (reused throughout the notebook) ---------------

def make_train_env(env_cls=HvacEnv, n_envs=4):
    """4 parallel envs, wrapped in VecNormalize (normalises observations AND
    rewards using running statistics gathered during training)."""
    venv = make_vec_env(env_cls, n_envs=n_envs)
    return VecNormalize(venv, norm_obs=True, norm_reward=True, clip_obs=10.0)


def make_eval_env(train_venv, env_cls=HvacEnv, seed=99):
    """Eval env that SHARES the training observation statistics (so it
    normalises the same way) but freezes them and reports the *raw* reward,
    so eval numbers are directly comparable to the random baseline."""
    eval_venv = make_vec_env(env_cls, n_envs=1, seed=seed)
    eval_venv = VecNormalize(eval_venv, norm_obs=True, norm_reward=False,
                             clip_obs=10.0, training=False)
    eval_venv.obs_rms = train_venv.obs_rms   # share running stats by reference
    return eval_venv


def rollout(model, venv, env_cls=HvacEnv, seed=7):
    """Run ONE deterministic episode on a fresh raw env, normalising each
    observation through `venv` before predict(). Returns per-step lists."""
    env = env_cls(seed=seed)
    obs, _ = env.reset()
    temps, powers, prices, hours = [], [], [], []
    done, hour = False, 0.0
    while not done:
        action, _ = model.predict(venv.normalize_obs(obs), deterministic=True)
        obs, _, term, trunc, info = env.step(action)
        done = term or trunc
        temps.append(info["room_temp"]); powers.append(info["heater_kw"])
        prices.append(info["price_eur_kwh"]); hour += 0.25; hours.append(hour)
    return hours, temps, powers, prices

In [ ]:
# Normalised, vectorised training env (4 parallel envs)
train_env = make_train_env(HvacEnv, n_envs=4)

# Separate eval env that shares the normalisation stats and reports raw reward
eval_env = make_eval_env(train_env, HvacEnv, seed=99)

# Evaluation callback: saves the best model automatically
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./logs/hvac_a2c/",
    eval_freq=2000,
    n_eval_episodes=5,
    verbose=0,
)

# Create and train the agent.
#   n_steps=16 + gae_lambda=0.95 -> lower-variance advantage estimates than the
#       very short default rollout (n_steps=5, gae_lambda=1.0).
#   normalize_advantage=True     -> standardises the advantage before the policy
#       update, which (together with VecNormalize) keeps the gradients well-scaled.
model = A2C(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=7e-4,
    n_steps=16,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.01,          # entropy bonus for exploration
    vf_coef=0.5,
    normalize_advantage=True,
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=200_000, callback=eval_callback)
print("Training complete.")

In [ ]:
# Evaluate the trained agent (raw reward, directly comparable to the baseline)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"Trained A2C agent: {mean_reward:.1f} ± {std_reward:.1f} per episode")

# Compare with a random baseline
def random_agent_reward(n_episodes=20):
    rewards = []
    for ep in range(n_episodes):
        env = HvacEnv(seed=ep)
        obs, _ = env.reset()
        total, done = 0.0, False
        while not done:
            obs, r, term, trunc, _ = env.step(env.action_space.sample())
            total += r
            done = term or trunc
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

rand_mean, rand_std = random_agent_reward()
print(f"Random agent     : {rand_mean:.1f} ± {rand_std:.1f} per episode")
improvement = (mean_reward - rand_mean) / abs(rand_mean) * 100
print(f"Improvement      : {improvement:+.1f} %")

In [ ]:
# Visualise the trained agent's behaviour over one day.
# `rollout` normalises each observation through train_env's stats before predict.
hours, temps, powers, prices = rollout(model, train_env, HvacEnv, seed=7)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(hours, temps, color="tomato")
axes[0].axhline(21, color="green", linestyle="--", label="Target 21°C")
axes[0].fill_between(hours, 20, 22, alpha=0.15, color="green")
axes[0].set_ylabel("Temperature (°C)")
axes[0].set_title("Trained A2C Agent — One Day")
axes[0].legend()

axes[1].bar(hours, powers, width=0.2, color="steelblue")
axes[1].set_ylabel("Heater power (kW)")

axes[2].plot(hours, prices, color="orange")
axes[2].set_ylabel("Electricity price (€/kWh)")
axes[2].set_xlabel("Hour of day")

plt.tight_layout()
plt.show()

❓ **Discuss:** Does the trained agent pre-heat when electricity is cheap? How does it respond to the price peak in the afternoon? Compare this to what you'd expect from a naive thermostat.

In [ ]:
#@title 💡 Sample answer — does it pre-heat when cheap? (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**Compare the heater-power panel against the price panel.** With the cost term weighted (`cost_weight=10`), a well-trained A2C agent should cluster its heating in the **low-price hours** — heating harder during the cheap overnight/early-morning trough and easing off through the evening price peak, riding the thermal mass. That is the load-shifting behaviour a naive thermostat can never produce, because a thermostat reacts only to temperature and buys whatever electricity costs at that moment.

**What to look for:**
- Heater bursts concentrated in cheap hours, a lull during the peak.
- Room temperature allowed to drift *slightly* above 21 °C before the peak (stored heat) and *slightly* below during it.

**Be honest about limits:** when the outdoor temperature is very cold, heat loss can exceed what pre-heating buffers, so the agent may have to heat through the peak anyway. And A2C is stochastic — some runs discover price-shifting more cleanly than others. Describe what *your* plot actually shows, not the idealised story.
"""))

In [ ]:
ent_coef_values = [0.0, 0.01, 0.1]  # feel free to add more
results = {}

for ent_coef in ent_coef_values:
    print(f"Training with ent_coef={ent_coef}...")
    train_env_hp = make_train_env(HvacEnv, n_envs=4)
    eval_env_hp = make_eval_env(train_env_hp, HvacEnv, seed=99)

    m = A2C("MlpPolicy", train_env_hp, learning_rate=7e-4, n_steps=16,
            gamma=0.99, gae_lambda=0.95, ent_coef=ent_coef,
            normalize_advantage=True, verbose=0, seed=42)
    m.learn(total_timesteps=100_000)

    mean, std = evaluate_policy(m, eval_env_hp, n_eval_episodes=10)
    results[ent_coef] = (mean, std)
    print(f"  ent_coef={ent_coef}: mean reward = {mean:.1f} ± {std:.1f}")

print("\n--- Summary ---")
for k, (m, s) in sorted(results.items()):
    print(f"ent_coef={k:4}: {m:.1f} ± {s:.1f}")

❓ **Reflect:**
1. Which `ent_coef` performed best?
2. What happens with `ent_coef=0`? Can the agent still learn?
3. What's the risk of very high entropy (e.g., `ent_coef=1.0`)?

*Bonus:* Also try varying `learning_rate` (e.g., 1e-3, 7e-4, 1e-4) and observe the effect.

In [ ]:
#@title 💡 Sample answer — entropy coefficient (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**1. Which `ent_coef` performed best?** Usually the small bonus (**≈ 0.01**). It keeps the policy exploring early without distracting it from the actual objective.

**2. What happens at `ent_coef = 0`?** The agent *can* still learn — A2C has no other exploration mechanism, so it relies on the natural stochasticity of its policy. But with no entropy bonus it is prone to **premature convergence**: it can lock onto a mediocre deterministic policy early, and results become more seed-dependent (lucky seeds fine, unlucky seeds stuck).

**3. Risk of very high entropy (e.g. 1.0)?** The entropy bonus then **dominates the return** — the agent is rewarded mainly for *acting randomly*, so it never commits to a good policy and performance stays poor. Exploration is a means, not the goal.

**Bonus — `learning_rate`:** too high (e.g. 0.1) → unstable, the policy can diverge or collapse; too low (e.g. 1e-5) → painfully slow. A2C's sweet spot here is around **7e-4**.
"""))

In [ ]:
from envs.hvac_env import HvacEnv

class HvacEnvCustomReward(HvacEnv):
    """Your custom reward variant."""

    def step(self, action: int):
        obs, reward, terminated, truncated, info = super().step(action)

        # TODO: Replace the reward with your custom version.
        # Useful values from info:
        #   info["room_temp"]     — current room temperature after heater
        #   info["price_eur_kwh"] — current electricity price
        #   info["heater_kw"]     — power used this step
        #   self._hour            — current hour (0.0–24.0)

        custom_reward = reward  # <-- replace this

        return obs, custom_reward, terminated, truncated, info


# Train your custom agent (same normalised setup as Part 3)
train_env_custom = make_train_env(HvacEnvCustomReward, n_envs=4)
eval_env_custom = make_eval_env(train_env_custom, HvacEnvCustomReward, seed=99)

model_custom = A2C("MlpPolicy", train_env_custom, learning_rate=7e-4, n_steps=16,
                   gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
                   normalize_advantage=True, verbose=0, seed=42)
model_custom.learn(total_timesteps=150_000)

mean_custom, std_custom = evaluate_policy(model_custom, eval_env_custom, n_eval_episodes=20)
print(f"Custom reward agent: {mean_custom:.1f} ± {std_custom:.1f}")

In [ ]:
#@title Example Solution — Night-mode reward (Option C, click ▶ to expand) {display-mode: "form"}
from envs.hvac_env import HvacEnv

class HvacEnvNightMode(HvacEnv):
    """Comfort matters less while occupants sleep (23:00-06:00)."""
    NIGHT_COMFORT_SCALE = 0.25   # down-weight discomfort at night

    def step(self, action: int):
        obs, reward, terminated, truncated, info = super().step(action)

        # super().step() has already advanced self._hour by one step, so the
        # hour at which THIS action was taken is one step earlier:
        decision_hour = (self._hour - self._DT) % 24.0
        is_night = decision_hour >= 23.0 or decision_hour < 6.0
        comfort_scale = self.NIGHT_COMFORT_SCALE if is_night else 1.0

        # Rebuild the reward from its raw components (info gives both).
        # Reuse self.cost_weight so the cost term matches the base env.
        custom_reward = -(comfort_scale * info["discomfort"]
                          + self.cost_weight * info["cost_eur"])
        return obs, custom_reward, terminated, truncated, info


# Train the night-mode agent (same normalised setup as Part 3)
train_env_night = make_train_env(HvacEnvNightMode, n_envs=4)
eval_env_night = make_eval_env(train_env_night, HvacEnvNightMode, seed=99)

model_custom = A2C("MlpPolicy", train_env_night, learning_rate=7e-4, n_steps=16,
                   gamma=0.99, gae_lambda=0.95, ent_coef=0.01,
                   normalize_advantage=True, verbose=0, seed=42)
model_custom.learn(total_timesteps=150_000)

mean_custom, std_custom = evaluate_policy(model_custom, eval_env_night, n_eval_episodes=20)
print(f"Night-mode agent reward: {mean_custom:.1f} ± {std_custom:.1f}")

# Visualise: does it let the room drift overnight (purple band) and tighten up by day?
# Evaluate on the ORIGINAL dynamics; normalise obs through the night-mode stats.
hours, temps, powers, _ = rollout(model_custom, train_env_night, HvacEnv, seed=7)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hours, temps, color="tomato", label="Room temp (night-mode agent)")
ax.axhline(21, color="green", linestyle="--", label="Target 21 °C")
ax.axvspan(0, 6, alpha=0.12, color="purple", label="Night (comfort relaxed)")
ax.axvspan(23, 24, alpha=0.12, color="purple")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Temperature (°C)")
ax.set_title("Night-mode agent — comfort is cheap to violate overnight")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

print("Expect the room to sag below 21 C overnight (cheap to be uncomfortable),")
print("then be pulled back toward 21 C during the day.")

❓ **Discuss:** How did changing the reward function change the agent's behaviour? Compare the temperature curves. In a real building, who gets to decide what the reward function should be — the ML engineer or the building manager?

In [ ]:
#@title 💡 Sample answer — how did changing the reward change behaviour? (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
Changing the reward changes the agent's **priorities**, not its competence — it still optimises perfectly, just for a different target. Concretely:

- **Comfort-first** → hugs 21 °C tightly, spends more on electricity.
- **Cost-first** → tolerates a wider temperature band to dodge the price peak.
- **Night-mode** → lets the room sag overnight (nobody's awake) and tightens up during the day.

Compare the temperature curves of two variants — the *shape* of the policy visibly shifts.

**Who decides the reward?** The **building manager / business owner**, not the ML engineer. The comfort-vs-cost weighting is a **business value judgement**, not a technical one. The engineer's job is to make that trade-off *explicit and tunable* and to show stakeholders the resulting behaviour. This is the essence of **reward engineering**: the agent will exploit exactly what you wrote down — including loopholes you didn't intend — so specify the objective carefully.
"""))

In [ ]:
# A deliberately bad training run — high learning rate, no entropy.
# NOTE: we still wrap in VecNormalize (same as the good agent), so the ONLY
# difference is the hyperparameters — that's what makes this a fair "what went
# wrong?" comparison rather than a normalisation artefact.
train_env_bad = make_train_env(HvacEnv, n_envs=4)
bad_model = A2C(
    "MlpPolicy",
    train_env_bad,
    learning_rate=0.1,   # way too high
    ent_coef=0.0,        # no exploration
    n_steps=16,
    gamma=0.99,
    gae_lambda=0.95,
    normalize_advantage=True,
    verbose=0,
    seed=42,
)
bad_model.learn(total_timesteps=50_000)

# Evaluate the bad agent (raw reward, comparable to the random baseline)
eval_env_bad = make_eval_env(train_env_bad, HvacEnv, seed=5)
mean_bad, std_bad = evaluate_policy(bad_model, eval_env_bad, n_eval_episodes=20)

# Recompute the GOOD agent here so this cell is self-contained (independent of
# whatever `temps`/`mean_reward` happen to hold after Parts 4–5).
eval_env_good = make_eval_env(train_env, HvacEnv, seed=5)
mean_good, std_good = evaluate_policy(model, eval_env_good, n_eval_episodes=20)

print(f"Bad agent reward : {mean_bad:.1f} ± {std_bad:.1f}")
print(f"Good agent reward: {mean_good:.1f} ± {std_good:.1f}")

# Roll out both agents on the SAME day (seed=7), normalising obs through each
# agent's own stats.
hours_bad,  temps_bad,  powers_bad,  _ = rollout(bad_model, train_env_bad, HvacEnv, seed=7)
hours_good, temps_good, powers_good, _ = rollout(model,     train_env,     HvacEnv, seed=7)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(hours_good, temps_good, color="green", linewidth=2, label="Good agent")
ax1.plot(hours_bad,  temps_bad,  color="red",   linewidth=2, label="Bad agent")
ax1.axhline(21, color="black", linestyle="--", label="Target 21°C")
ax1.fill_between(hours_good, 20, 22, alpha=0.1, color="green")
ax1.set_ylabel("Temperature (°C)")
ax1.legend()
ax1.set_title("Bad vs. Good Agent")

ax2.bar(hours_bad,  powers_bad,  width=0.2, color="red",   alpha=0.7, label="Bad agent")
ax2.bar(hours_good, powers_good, width=0.2, color="green", alpha=0.5, label="Good agent")
ax2.set_ylabel("Heater power (kW)")
ax2.set_xlabel("Hour of day")
ax2.legend()
plt.tight_layout()
plt.show()

❓ **Diagnostic questions:**
1. What action does the bad agent take most of the time? Can you see it in the chart?
2. Which of the two hyperparameter problems (`learning_rate=0.1` or `ent_coef=0.0`) do you think is more damaging? Why?
3. What would you change first to fix it? Try it!

**Tip:** An agent that always picks the same action may have:
- Converged too early to a suboptimal policy (no exploration)
- Gradient exploded due to too-high learning rate, causing random weights

These look similar from the outside but have different fixes.

In [ ]:
#@title 💡 Sample answer — diagnosing the bad agent (click ▶ to expand) {display-mode: "form"}
from IPython.display import Markdown, display
display(Markdown(r"""
**1. What does the bad agent do?** It typically **collapses to a single action** — e.g. heater permanently OFF (or stuck on one power level). You can see it in the chart: a flat heater-power bar and the temperature drifting away from 21 °C instead of being regulated.

**2. Which hyperparameter is more damaging — `learning_rate=0.1` or `ent_coef=0.0`?** Here the **learning rate** is the bigger culprit. At 0.1 the gradient updates massively overshoot, destabilising the actor and critic networks and pushing the policy into a degenerate corner. `ent_coef=0` on its own only removes the exploration bonus — riskier, but A2C can still learn. The two together are worst-case.

**3. What to fix first?** **Lower the learning rate** (e.g. back to 7e-4); then add a little entropy back (`ent_coef=0.01`) and retrain. Try it — you should see the agent start regulating temperature again.

**Why this is subtle:** "always one action" has *two* possible causes that look identical from the outside — (a) **premature convergence** from too little exploration, or (b) **exploded gradients** from too high a learning rate scrambling the weights. The fixes are opposite (more entropy vs. lower lr), so diagnosing *which* failure you have is the real skill. Here, the obvious 0.1 learning rate points to (b).
"""))

## Summary

| What you did | RL Engineer skill |
|---|---|
| Explored HvacEnv before training | Always understand your environment first |
| Trained A2C with SB3 in 5 lines | Use production tools instead of reinventing |
| Compared trained vs. random agent | Always have a baseline |
| Swept `ent_coef` values | Hyperparameter sensitivity analysis |
| Customised the reward function | Reward engineering — the hardest real-world RL skill |
| Diagnosed the bad agent | Debugging is 50% of the job |

**Next:** In the PPO notebook, you'll apply similar skills to a **dynamic pricing** problem.